In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import html


In [18]:
countries = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czechia",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Iceland", "Ireland", "Italy", "Latvia", "Lithuania",
    "Luxembourg", "Netherlands", "Norway", "Malta", "Poland", "Portugal", "Romania",
    "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "United Kingdom"
]

In [19]:
links = []

for country in countries:
    country_no_space = country.replace(" ", "").lower()
    links.append(f"http://www.parties-and-elections.eu/{country_no_space}.html")


links

['http://www.parties-and-elections.eu/austria.html',
 'http://www.parties-and-elections.eu/belgium.html',
 'http://www.parties-and-elections.eu/bulgaria.html',
 'http://www.parties-and-elections.eu/croatia.html',
 'http://www.parties-and-elections.eu/cyprus.html',
 'http://www.parties-and-elections.eu/czechia.html',
 'http://www.parties-and-elections.eu/denmark.html',
 'http://www.parties-and-elections.eu/estonia.html',
 'http://www.parties-and-elections.eu/finland.html',
 'http://www.parties-and-elections.eu/france.html',
 'http://www.parties-and-elections.eu/germany.html',
 'http://www.parties-and-elections.eu/greece.html',
 'http://www.parties-and-elections.eu/hungary.html',
 'http://www.parties-and-elections.eu/iceland.html',
 'http://www.parties-and-elections.eu/ireland.html',
 'http://www.parties-and-elections.eu/italy.html',
 'http://www.parties-and-elections.eu/latvia.html',
 'http://www.parties-and-elections.eu/lithuania.html',
 'http://www.parties-and-elections.eu/luxembourg.

In [22]:
trial_links = ['http://www.parties-and-elections.eu/bulgaria.html']

In [31]:
for url in trial_links:
    
    name = re.search(r'/([^/]+)\.html$', url).group(1)

    response = requests.get(url)

    response.encoding = 'windows-1252'

    soup = BeautifulSoup(response.text, "html.parser")

    tables = soup.find_all("table")

    
    tables = soup.find_all("table")
        
    target_table = tables[5]

    tables[5].get_text()

    rows = target_table.find_all("tr")

    table_values = []

    for row in rows:
        cells = row.find_all("td")
        values = [c.get_text(strip=True) for c in cells]
        table_values.append(values)
    
    print(table_values)
    
    header_call_number = len(table_values[1]) + 1

    party_information_raw = []

    for value in table_values:
        if len(value) == header_call_number: 
            party_information_raw.append(value)
        elif re.search(r'\(.*?\)', value[1]):
            party_information_raw.append(["", value[1]] + [""] * (header_call_number - 2))
    
    print(party_information_raw)

[['', '2026', '', '2024II', '', '2024I', '', '2023'], ['Party', 'Ideology', '%', 'Seats', '%', 'Seats', '%', 'Seats', '%', 'Seats'], ['', 'Graždani za Evropejsko Razvitie na Bălgaria (GERB)Citizens\r\n  for European Development of Bulgaria', 'Conservatism', '', '', '26,4', '65', '24,0', '66', '26,5', '67'], ['', 'Săjuz na Demokratični\r\n  Sili (SDS)Union\r\n  of Democratic Forces', 'ConservatismChristian democracy', '', '', '1', '2', '2'], ['', 'Prodŭlžavame Promjanata (PP)We\r\n  Continue the Change', 'Anti-corruption\r\n  politicsLiberalism', '', '', '14,2', '19', '13,9', '22', '24,5', '42'], ['', 'Demokratična Bălgaria (DB)Democratic\r\n  Bulgaria', 'LiberalismConservatism', '', '', '17', '17', '22'], ['', 'Văzraždane (V)Revival', 'National\r\n  conservatism', '', '', '13,4', '33', '13,4', '38', '14,2', '37'], ['', 'Dviženie za Prava i\r\n  Svobodi (DPS)Movement\r\n  for Rights and Freedoms', 'Minority\r\n  interests (TUR)Centrism', '', '', '11,5', '29', '16,6', '47', '13,7', '36']

In [20]:
election_data = {}


for url in links:
    
    name = re.search(r'/([^/]+)\.html$', url).group(1)

    response = requests.get(url)

    response.encoding = 'windows-1252'

    soup = BeautifulSoup(response.text, "html.parser")

    tables = soup.find_all("table")

    try:
    
        tables = soup.find_all("table")
        
        target_table = tables[5]

        tables[5].get_text()

        rows = target_table.find_all("tr")

        table_values = []

        for row in rows:
            cells = row.find_all("td")
            values = [c.get_text(strip=True) for c in cells]
            table_values.append(values)
        
        header_call_number = len(table_values[1]) + 1

        party_information_raw = []

        for value in table_values:
            if len(value) == header_call_number: 
                party_information_raw.append(value)
            elif re.search(r'\(.*?\)', value[1]):
                party_information_raw.append(["", value[1]] + [""] * (header_call_number - 2))

        party_names = []
        election_results_all = []


        for value in party_information_raw: 
            party_names.append(value[1])
            election_results_all.append(value[3:])


        year_row = table_values[0]

        election_years = [v for v in year_row if re.search(r'\d+', v)]

        election_results_percent = []
        for result in election_results_all:
            election_results_percent.append(result[::2])

        cleaned_name = []
        name_short = []

        for party in party_names:
            clean = party.replace("\n", " ").replace("\r", " ").replace("\\", "").replace(" +", " ")

            clean = re.sub(r'\s+', ' ', clean)

            match = re.match(r'(.*?)\(', clean)
            
            if match:
                cleaned_name.append(match.group(1).strip())
            else:
                cleaned_name.append(clean.strip())
    
            within_parenthesis = re.search(r'\((.*?)\)', clean)

            if within_parenthesis:
                name_short.append(within_parenthesis.group(1).strip())
            else:
                name_short.append("")

        df = pd.concat([
            pd.Series(cleaned_name, name="Party"),
            pd.Series(name_short, name="Name_Short"),
            pd.DataFrame(election_results_percent, columns = election_years)
            ], axis = 1)

        election_data[name] = df
    
    except:
        print(name)

In [42]:
country_codes = {
    "austria": "AUT", "belgium": "BEL", "bulgaria": "BGR", "croatia": "HRV",
    "cyprus": "CYP", "czechia": "CZE", "denmark": "DNK", "estonia": "EST",
    "finland": "FIN", "france": "FRA", "germany": "DEU", "greece": "GRC",
    "hungary": "HUN", "iceland": "ISL", "ireland": "IRL", "italy": "ITA",
    "latvia": "LVA", "lithuania": "LTU", "luxembourg": "LUX", "malta": "MLT", "netherlands": "NLD",
    "norway": "NOR", "poland": "POL", "portugal": "PRT", "romania": "ROU",
    "slovakia": "SVK", "slovenia": "SVN", "spain": "ESP", "sweden": "SWE",
    "switzerland": "CHE", "unitedkingdom": "GBR"
}

for country_name, df in election_data.items():
    code = country_codes.get(country_name) 
    df["CountryCode"] = code

relevant_columns = ["Party", "Name_Short","2026", "2025", "2024", "2023", "2022", "CountryCode"]

for country_name, df in election_data.items():
    election_data[country_name] = df[[
        col for col in df.columns
        if any(term in col for term in relevant_columns)
    ]]

In [43]:
election_data["austria"]

,Party,Name_Short,2024,CountryCode
0,freiheitliche partei osterreichs,fpo,"28,8",AUT
1,osterreichische volkspartei,ovp,"26,3",AUT
2,sozialdemokratische partei osterreichs,spo,"21,1",AUT
3,das neue osterreich und liberales forum,neos,"9,1",AUT
4,die grunen die grune alternative,grune,"8,2",AUT
5,kommunistische partei osterreichs kpo plus,kpo,"2,4",AUT
6,jetzt liste pilz,jetzt,-,AUT
7,team stronach fur osterreich,ts,-,AUT


In [ ]:
import string
import unicodedata

def remove_accents(text):
    if pd.isna(text):
        return text  # keep NaN as-is
    text = str(text)  # ensure it is a string
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))

test_data = election_data.copy()

for k, v in test_data.items():
    v["Party"] = v["Party"].str.lower()
    v["Name_Short"] = v["Name_Short"].str.lower()
    # replace all non-letter characters with a space
    punctuation_regex = r'[!"\\#\$%\&\'\(\)\*\+,\-\./:;<=>\\?@\[\\\]\^_`\{\|\}\~\u2010-\u2015]'
    v["Party"] = v["Party"].str.replace(punctuation_regex, " ", regex=True)
    v["Name_Short"] = v["Name_Short"].str.replace(punctuation_regex, " ", regex=True)

    # replace multiple spaces with a single space and strip leading/trailing spaces
    v["Party"] = v["Party"].str.replace(r'\s+', ' ', regex=True).str.strip()
    v["Name_Short"] = v["Name_Short"].str.replace(r'\s+', ' ', regex=True).str.strip()

    
    v["Party"] = v["Party"].apply(remove_accents)
    v["Name_Short"] = v["Name_Short"].apply(remove_accents)



test_data["austria"]

'[!"\\\\#\\$%\\&\\\'\\(\\)\\*\\+,\\-\\./:;<=>\\\\?@\\[\\\\\\]\\^_`\\{\\|\\}\\~\\u2010-\\u2015]'

In [49]:
id_frame = pd.read_csv('/Users/lukefischer/Dropbox/The PopuList Repo/Data/populist_parlgov_id.csv')

id_frame = id_frame[id_frame["CountryCode"].isin(country_codes.values())]

id_frame["Party"] = id_frame["Party"].str.lower()
id_frame["Name_Short_y"] = id_frame["Name_Short_y"].str.lower()
id_frame["Name_Short_x"] = id_frame["Name_Short_x"].str.lower()
    
# replace all non-letter characters with a space
id_frame["Party"] = id_frame["Party"].apply(remove_accents)
id_frame["Name_Short_y"] = id_frame["Name_Short_y"].apply(remove_accents)
id_frame["Name_Short_x"] = id_frame["Name_Short_x"].apply(remove_accents)

# now ü → u, so the regex won't destroy it
id_frame["Party"] = id_frame["Party"].str.replace(punctuation_regex, " ", regex=True)
id_frame["Name_Short_y"] = id_frame["Name_Short_y"].str.replace(punctuation_regex, " ", regex=True)
id_frame["Name_Short_x"] = id_frame["Name_Short_x"].str.replace(punctuation_regex, " ", regex=True)

id_frame["Party"] = id_frame["Party"].str.replace(r'\s+', ' ', regex=True).str.strip()
id_frame["Name_Short_y"] = id_frame["Name_Short_y"].str.replace(r'\s+', ' ', regex=True).str.strip()
id_frame["Name_Short_x"] = id_frame["Name_Short_x"].str.replace(r'\s+', ' ', regex=True).str.strip()


In [50]:
id_frame

,Party,Name_Short_y,Name_Short_x,CountryCode,parlgov_id
0,bundnis zukunft osterreich,bzo,bzo,AUT,1536
1,freiheitliche partei osterreichs,fpo,fpo,AUT,50
2,jetzt liste pilz,pilz,pilz,AUT,2651
3,team stronach,ts,ts,AUT,2150
4,nieuw vlaamse alliantie,n va,n va,BEL,501
...,...,...,...,...,...
140,slovenska demokratska stranka,sds,sds,SVN,179
141,slovenska nacionalna stranka,sns,sns,SVN,981
142,zdruzena lista socialni demokrati,zl sd,zl,SVN,706
143,sverigedemokraterna,sd,sd,SWE,1546


In [57]:
id_elections = {}

for k, v in test_data.items():
    merged = v.merge(
        id_frame,
        how="left",                  
        on=["CountryCode", "Party"],
        )
    merged = merged.drop_duplicates()
    
    id_elections[k] = merged

for k, v in test_data.items():
    merged = v.merge(
        id_frame,
        how="left",                  
        left_on=["CountryCode", "Name_Short"],
        right_on=["CountryCode", "Name_Short_x"],
    )
    merged = merged.drop_duplicates()
    
    id_elections[k] = merged

In [58]:
id_elections["czechia"]

,Party_x,Name_Short,2025,CountryCode,Party_y,Name_Short_y,Name_Short_x,parlgov_id
0,ano 2011,ano,"34,5",CZE,akce nespokojenych obcanu 2011,ano,ano,2263.0
1,obcanska demokraticka strana,ods,"23,4",CZE,NaN,NaN,NaN,NaN
2,krest’anska a demokraticka unie,kdu csl,,CZE,NaN,NaN,NaN,NaN
3,top 09,top09,,CZE,NaN,NaN,NaN,NaN
4,starostove a nezavisli,stan,"11,2",CZE,NaN,NaN,NaN,NaN
5,ceska piratska strana,pirati,,CZE,NaN,NaN,NaN,NaN
6,svoboda a prima demokracie,spd,"7,8",CZE,svoboda a prima demokracie,spd,spd,2654.0
7,motoriste sobe,auto,"6,8",CZE,NaN,NaN,NaN,NaN
8,stacilo,stacilo,"4,3",CZE,NaN,NaN,NaN,NaN
9,ceska strana socialne demokraticka,cssd,STA,CZE,NaN,NaN,NaN,NaN


In [55]:
missing_id = []
missing_keys = []

for k, v in id_elections.items():
    missing_mask = v["parlgov_id"].isna()
    missing_id.extend(zip(v.loc[missing_mask, "Party"], v.loc[missing_mask, "CountryCode"]))
    missing_keys.extend(v.loc[missing_mask, "Party"].to_list())

missing_id

[('osterreichische volkspartei', 'AUT'),
 ('sozialdemokratische partei osterreichs', 'AUT'),
 ('das neue osterreich und liberales forum', 'AUT'),
 ('die grunen die grune alternative', 'AUT'),
 ('kommunistische partei osterreichs kpo plus', 'AUT'),
 ('team stronach fur osterreich', 'AUT'),
 ('vlaams belang', 'BEL'),
 ('mouvement reformateur', 'BEL'),
 ('parti du travail de belgique', 'BEL'),
 ('vooruit', 'BEL'),
 ('parti socialiste', 'BEL'),
 ('christen democratisch vlaams', 'BEL'),
 ('les engages', 'BEL'),
 ('open vlaamse liberalen en democraten', 'BEL'),
 ('de vlaamse groenen', 'BEL'),
 ('ecolo', 'BEL'),
 ('democrate federaliste independant', 'BEL'),
 ('lijst dedecker', 'BEL'),
 ('grazdani za evropejsko razvitie na balgaria', 'BGR'),
 ('sajuz na demokraticni sili', 'BGR'),
 ('produlzavame promjanata', 'BGR'),
 ('demokraticna balgaria', 'BGR'),
 ('vazrazdane', 'BGR'),
 ('dvizenie za prava i svobodi', 'BGR'),
 ('balgarska socialisticeska partija', 'BGR'),
 ('alians za prava i svobodi', 

In [ ]:
manual_id = {
'kommunistische partei osterreichs kpo plus': 996,
 'de vlaamse groenen': 528,
 'sajuz na demokraticni sili': 482,
 'produlzavame promjanata': 8994,
 'demokraticna balgaria': 9083,
 'vazrazdane': 5862,
 'alians za prava i svobodi': 'NaN',
 'moral edinstvo cest': 9812,
 'velicie': 9811,
 'dvizenie da balgarija': 2058,
 'zeleno dvizenie': 'NaN',
 'umirovljenici zajedno': 3918,
 'nezavisna platforma sjever': 'NaN',
 'hrvatski stranka prava dr ante starcevic': 3706,
 'dalija oreskovic i ljudi s imenom i prezimenom': 2524,
 'zeleno lijeva koalicija' : 'NaN', # not important
 'kinima oikologon sinergasia politon': 858,
 'krest’anska a demokraticka unie': 676,
 'motoriste sobe': 'NaN', # no ID
 'stacilo', # no ID
 'borgernes parti',
 'isamaa erakond',
 'nouveau front populaire',
 'les ecologistes europe ecologie les verts',
 'generations',
 'nouveau centre les centristes',
 'gauche republicaine et socialiste',
 'bundnis sahra wagenknecht',
 'panellinio sosialistiko kinima kinima allagis',
 'dimokratiko patriotiko kinima–niki',
 'laikos syndesmos chrysi avyi',
 'christianodimokratiko komma elladas',
 'anexartitoi ellinesnosi',
 'landesselbstverwaltung der ungarndeutschen',
 'osszefogas 2014',
 'independent ireland',
 'solidarity',
 '100 redress',
 'europa verde verdi',
 'noi con italia',
 'italia al centro',
 'sud chiama nord',
 'area civica',
 'direzione italia',
 'la sinistra l arcobaleno',
 'apvienotais saraksts',
 'apvieniba latvijai',
 'latvijai un ventspilij',
 'politine partija nemuno ausra',
 'nacionalinis susivienijimas',
 'tautos ir teisingumo sajunga',
 'partija laisve ir teisingumas',
 'politine partija drasos kelias',
 'letzebuergesch sozialistesch arbechterpartei',
 'ja 21',
 'boer burger beweging',
 '50 plus',
 'bij 1',
 'pasientfokus',
 'republikanie',
 'trzecia droga',
 'centrum dla polski',
 'nowa levica',
 'levica razem',
 'porozumienie jaroslawa gowina',
 'juntos pelo povo',
 'cds partido popular',
 'madeira primeiro',
 's o s romania',
 'partidul oamenilor tineri',
 'smer slovenska socialna demokracia',
 'szovetseg aliancia',
 'demokrati',
 'kotlebovci ludova strana nase slovensko',
 'strana madarskej komunity magyar kozosseg partja',
 'nova slovenija krscanskih demokratov',
 'fokus marka lotrica',
 'demokrati anzeta logarja',
 'drzavljansko gibanje resnica',
 'vesna–zelena stranka',
 'sumar',
 'eusko alderdi jeltzalea',
 'nueva canarias',
 'movimento sumar',
 'esquerra verda',
 'christlich soziale partei obwalden'}

In [ ]:
manual_id = {'kommunistische partei osterreichs kpo plus' : 996,
 'de vlaamse groenen' : 528,
 'produlzavame promjanata': 8994,
 'vazrazdane': 5862,
 'alians za prava i svobodi': 'NaN',
 'moral edinstvo cest': 9812,
 'velicie': 9811,